# Train a name detector for the bot

Trains the small model that hears the bot's name as a sound (`<name>.onnx` in `whisper/extras`), using [livekit-wakeword](https://github.com/livekit/livekit-wakeword) 0.3.0 (Apache-2.0) and synthetic voices, with two fixes to how it prepares its training clips (cell 3).

1. **Runtime > Change runtime type > T4 GPU** (free).
2. In cell 2, set `NAME` to the bot's name in lowercase (the same as `BOT_NAME`), other spellings of how people say it, and some words that only sound like it.
3. **Runtime > Run all** and keep this tab open. It takes about an hour. Cell 4 prints how well it did and whether it's usable; cell 5 downloads `<name>.onnx`. Save it as soon as it's done: Colab deletes its files when the session ends.
4. Put the file in the bot's `whisper/extras` folder and restart the bot. The start-up check shows a "Name detector" line.

Why the fixes: with its default settings, livekit-wakeword 0.3.0 puts exact silence before every clip of the name and around every other word, so the model can learn "a word right after silence" instead of the name. Babel's first model did just that: it scored "table", "label" and even "yeah" as high as "Babel", while its own test said 99%. It also gave the model its own clips and its downloaded examples of general speech at two different audio scales. Cell 3 fixes both, and cell 4 checks the name against the sound-alike words laid out the same way.

Cell 4 ends with the check that decides whether to use the file: the name and each sound-alike word on its own, scored the way the bot scores a sentence. If it says "Not usable", keep the file out of `whisper/extras`. A name that differs from common words by a single sound can fail it even after a good run: for Babel (BAY-bel), "fable", "label", "able" and "table" scored 0.89 to 0.95 against 0.97 for the name.


In [ ]:
# 1. Check the GPU and install the trainer (a few minutes)
!nvidia-smi -L || echo 'No GPU: Runtime > Change runtime type > T4 GPU, then run again'
!apt-get -qq update > /dev/null && apt-get -qq install -y espeak-ng ffmpeg sox libsndfile1 > /dev/null && echo 'system packages ok'
!pip -q install 'livekit-wakeword[train,eval,export]==0.3.0' && echo 'trainer installed'


In [ ]:
# 2. Settings, then the training data (about 16 GB, several minutes)
NAME = 'babel'  # the bot's name in lowercase, as in BOT_NAME
SPELLINGS = ['babble']  # other spellings of how people say it ("babble" for BAB-el)
SOUNDS_LIKE = ['bagel', 'label', 'table', 'cable', 'maple', 'fable', 'able', 'mabel', 'barbell', 'bible',
               'baby', 'babe', 'people', 'battle', 'paypal', 'rebel', 'bubble', 'bottle', 'level', 'apple']
# What people often say right after the name. They count as other words too, and they're
# what the fixes put right after some name clips ("Babel, what time is it?").
FOLLOW_UPS = ['what time is it', 'what', 'where are you', 'why', 'who won', 'how', 'you there', 'are you there',
              'can you', 'do you', 'did you', 'will you', 'is it', 'look at this', 'wait', 'yo', 'okay', 'yeah',
              'hey', 'are you awake']
# How people call it: the name alone, after "hey" or "yo", and running straight into the
# question with no pause (the model has to hear the name with the next word starting).
SAYS = [NAME, *SPELLINGS, f'hey {NAME}', f'yo {NAME}'] + [f'{NAME} {w}' for w in ('what', 'are you', 'you', 'can you', 'where')]

import json
config = f'''
model_name: {NAME}
target_phrases: {json.dumps(SAYS)}
custom_negative_phrases: {json.dumps(SOUNDS_LIKE + FOLLOW_UPS)}
n_samples: 10000
n_samples_val: 2000
steps: 50000
target_fp_per_hour: 0.2
max_negative_weight: 300  # the trainer raises it if false alarms stay high
model:
  model_type: conv_attention
  model_size: small
data_dir: ./data
output_dir: ./output
'''
open(f'{NAME}.yaml', 'w').write(config)
!livekit-wakeword setup --config {NAME}.yaml


In [ ]:
%%writefile wakeword_fixes.py
# Two fixes to livekit-wakeword 0.3.0's training, so the model learns the name
# itself rather than how the practice clips are laid out:
#
# 1. Audio scale. Its own clips reach the mel model as -1..1 floats, while the
#    ACAV100M and validation features it downloads (from openWakeWord) were made
#    from 16-bit sample values. The two sets then sit 9 units apart on every
#    feature, and the model can tell them apart by that alone. Here every clip
#    goes in as 16-bit values, like openWakeWord and like the bot's server.
# 2. Clip layout. It put silence (exact zeros) before every name clip and around
#    every other word, so "a word right after silence" meant "the name": the
#    first Babel model scored "table", "label" and even "yeah" at 0.95. Here the
#    other words are placed the same way as the name (and at random spots too),
#    and every clip gets background noise or a quiet noise floor all the way
#    through, so silence gives nothing away.
import random
import re

import numpy as np

from livekit.wakeword.data import augment as _augment
from livekit.wakeword.models import feature_extractor as _fe

SR = 16000
JITTER = 3200  # the name ends 0-200 ms before the end of the window (as livekit does)

if not getattr(_fe.MelSpectrogramFrontend, '_int16_scale', False):
    _original_forward = _fe.MelSpectrogramFrontend._forward_onnx

    def _forward_int16(self, audio):
        return _original_forward(self, np.asarray(audio, dtype=np.float32) * 32768.0)

    _fe.MelSpectrogramFrontend._forward_onnx = _forward_int16
    _fe.MelSpectrogramFrontend._int16_scale = True


def _place_end(x, n, rng):
    out = np.zeros(n, np.float32)
    end = n - rng.randint(0, JITTER)
    x = x[-end:]
    out[end - len(x):end] = x
    return out


def _place_anywhere(x, n, rng):
    """A random 2 s view of the clip: its start, middle or end."""
    visible = min(len(x), SR // 4)
    end = rng.randint(visible, n + len(x) - visible)
    start = end - len(x)
    out = np.zeros(n, np.float32)
    a, b = max(0, start), min(n, end)
    out[a:b] = x[a - start:b - start]
    return out


def _background(augmentor, n, rng):
    import soundfile as sf

    bg, _ = sf.read(str(rng.choice(augmentor.background_files)), dtype='float32')
    if bg.ndim > 1:
        bg = bg[:, 0]
    if len(bg) < n:
        bg = np.tile(bg, n // max(1, len(bg)) + 1)
    start = rng.randint(0, len(bg) - n)
    return bg[start:start + n]


def _noise_floor(n, rng):
    """White or brownish noise at -75 to -45 dBFS, like a quiet or noise-suppressed mic."""
    g = np.random.default_rng(rng.randrange(2**32))
    x = g.standard_normal(n).astype(np.float32)
    if rng.random() < 0.5:
        x = np.cumsum(x)
        x -= np.convolve(x, np.ones(400) / 400, mode='same')  # remove the drift
    x /= np.sqrt(np.mean(x ** 2)) + 1e-9
    return x * 10 ** (rng.uniform(-75, -45) / 20)


def _other_speech(clip_dir, rng, cache={}):
    """A random clip of other words from the same split (train or test), for context."""
    import soundfile as sf

    split = 'negative_test' if clip_dir.name.endswith('_test') else 'negative_train'
    folder = clip_dir.parent / split
    if folder not in cache:
        cache[folder] = sorted(p for p in folder.glob('clip_*.wav') if re.match(r'^clip_\d{6}\.wav$', p.name))
    if not cache[folder]:
        return None
    x, _ = sf.read(str(rng.choice(cache[folder])), dtype='float32')
    x = x[:, 0] if x.ndim > 1 else x
    return x / (np.abs(x).max() + 1e-9) * rng.uniform(0.5, 1.2)


def _with_context(window, clip_dir, rng, gain):
    """People rarely say a word alone: sometimes other speech runs up to it, or starts right
    after it (the window then ends just into the next word). Done the same way for the name
    and for other words, so only the word itself tells them apart."""
    n = len(window)
    loud = np.flatnonzero(np.abs(window) > 1e-4)
    if not len(loud):
        return window
    first, last = loud[0], loud[-1]
    if rng.random() < 0.5 and last < n - 400:
        after = _other_speech(clip_dir, rng)
        if after is not None:
            start = last + rng.randint(0, 4000)  # 0-250 ms later
            take = after[: max(0, n - start)] * gain
            window[start:start + len(take)] += take
    if rng.random() < 0.4 and first > 1600:
        before = _other_speech(clip_dir, rng)
        if before is not None:
            end = first - rng.randint(800, 6400)  # ends 50-400 ms before
            take = before[-end:] * gain if end > 0 else before[:0]
            window[end - len(take):end] += take
    return window


def _fill(window, speech_rms, augmentor, rng):
    n = len(window)
    if augmentor.background_files and rng.random() < 0.7:
        bg = _background(augmentor, n, rng)
        bg_rms = np.sqrt(np.mean(bg ** 2)) + 1e-9
        window = window + bg * (speech_rms / bg_rms) * 10 ** (-rng.uniform(0, 30) / 20)
    window = window + _noise_floor(n, rng)
    return np.clip(window, -1, 1).astype(np.float32)


def _augment_directory(clip_dir, augmentor, is_positive, target_duration_s=2.0, sample_rate=16000, round_idx=0):
    import soundfile as sf
    from tqdm import tqdm

    rng = random.Random(f'{clip_dir.name}-{round_idx}')
    n = int(target_duration_s * sample_rate)
    is_background = clip_dir.name.startswith('background')
    source = re.compile(r'^clip_\d{6}\.wav$') if round_idx == 0 else re.compile(rf'^clip_\d{{6}}_r{round_idx - 1}\.wav$')
    files = sorted(p for p in clip_dir.glob('*.wav') if source.match(p.name))
    for path in tqdm(files, desc=f'Augmenting {clip_dir.name} r{round_idx}', unit='clip'):
        audio, _ = sf.read(str(path), dtype='float32')
        if audio.ndim > 1:
            audio = audio[:, 0]
        if round_idx == 0 and not is_background:
            speech = augmentor.apply_rir(augmentor.augment_clip(audio))
            gain = 10 ** (rng.uniform(-24, -1) / 20)
            speech = speech / (np.abs(speech).max() + 1e-9) * gain
            if is_positive or rng.random() < 0.5:
                window = _with_context(_place_end(speech, n, rng), clip_dir, rng, gain)
            else:
                window = _place_anywhere(speech, n, rng)
            speech_rms = np.sqrt(np.mean(speech ** 2)) + 1e-9
        else:
            # Background clips (and later rounds): already noise or already placed.
            window = np.zeros(n, np.float32)
            take = audio[:n]
            window[:len(take)] = take
            speech_rms = np.sqrt(np.mean(take ** 2)) + 1e-9
        window = _fill(window, speech_rms, augmentor, rng)
        stem = re.sub(r'_r\d+$', '', path.stem)
        sf.write(str(path.with_name(f'{stem}_r{round_idx}.wav')), window, sample_rate)


_augment._augment_directory = _augment_directory
print('Training fixes applied: 16-bit audio scale, same layout for the name and other words, no silent padding.')


In [ ]:
# 4. Make the voices, train and check it (about 45 minutes)
# Each step runs here rather than with `livekit-wakeword run`: in version 0.3.0 that
# command stops at the feature step (it calls run_extraction without its second
# argument). Running this cell again picks up the voices it already made.
import hashlib, logging, sys
import numpy as np
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', stream=sys.stdout, force=True)
import wakeword_fixes  # cell 3, before anything is augmented or measured
from livekit.wakeword.config import load_config
from livekit.wakeword.data.augment import run_augment
from livekit.wakeword.data.features import run_extraction
from livekit.wakeword.data.generate import run_generate
from livekit.wakeword.eval.evaluate import _predict_onnx, run_eval
from livekit.wakeword.export.onnx import run_export
from livekit.wakeword.training.trainer import run_train

cfg = load_config(f'{NAME}.yaml')
run_generate(cfg)
run_augment(cfg)
run_extraction(cfg, None)
run_train(cfg)
run_export(cfg)
model_path = cfg.model_output_dir / f'{NAME}.onnx'
results = run_eval(cfg, model_path)

# The check that matters: test clips of the name against test clips of the sound-alike
# words, prepared the same way, so layout can't give the answer away.
import onnxruntime as ort
session = ort.InferenceSession(str(model_path), providers=['CPUExecutionProvider'])
pos = _predict_onnx(session, np.load(cfg.model_output_dir / 'positive_features_test.npy'))
neg = _predict_onnx(session, np.load(cfg.model_output_dir / 'negative_features_test.npy'))
for t in (0.5, 0.7):
    print(f'At a score of {t}: heard {np.mean(pos >= t):.1%} of test calls of its name, '
          f'and mistook {np.mean(neg >= t):.1%} of the other test words for it.')
t = float(np.quantile(neg, 0.99))
print(f'A threshold of {t:.2f} lets through 1% of the other words and hears {np.mean(pos > t):.1%} of the name calls.')
print(f"Over {results['validation_hours']:.1f} hours of other speech: {results['fpph']:.2f} false alarms per hour (at 0.5).")
print('Model file:', model_path, hashlib.sha256(model_path.read_bytes()).hexdigest())

CHECK_WORDS, N_CHECK = SOUNDS_LIKE, 24
# Real calls on Discord are often noise-suppressed, so also check the name and each
# sound-alike word on its own over a quiet floor, scored the way the bot's server scores
# a sentence (the best of many 2 s windows). Words that only sound like the name must
# score well below it, or the bot would react to them.
import pathlib, shutil
import soundfile as sf
from livekit.wakeword.data.tts import get_tts_backend
from livekit.wakeword.models.feature_extractor import MelSpectrogramFrontend, SpeechEmbedding
from livekit.wakeword.resources import get_embedding_model_path, get_mel_model_path

tts = get_tts_backend(cfg)
mel_frontend = MelSpectrogramFrontend(get_mel_model_path())
speech_embedding = SpeechEmbedding(get_embedding_model_path())

def server_score(clip):
    floor = np.random.default_rng(len(clip)).standard_normal(32000 + len(clip) + 5333) * 1e-3
    x = floor.astype(np.float32)
    x[32000:32000 + len(clip)] += clip
    emb = speech_embedding.extract_embeddings(mel_frontend(x))[0]
    windows = np.stack([emb[i:i + 16] for i in range(len(emb) - 15)]).astype(np.float32)
    return float(_predict_onnx(session, windows).max())

check_dir = pathlib.Path('check_clips')
shutil.rmtree(check_dir, ignore_errors=True)
medians = {}
for word in [NAME] + CHECK_WORDS:
    folder = check_dir / word.replace(' ', '_')
    tts.synthesize_clips(phrases=[word], output_dir=folder, n_samples=N_CHECK, start_index=0, batch_size=N_CHECK)
    scores = []
    for path in sorted(folder.glob('clip_*.wav')):
        audio, _ = sf.read(str(path), dtype='float32')
        scores.append(server_score(audio / (np.abs(audio).max() + 1e-9) * 0.5))
    medians[word] = float(np.median(scores))
others = {w: s for w, s in medians.items() if w != NAME}
worst = max(others, key=others.get)
print('Each word on its own over a quiet floor (median of', N_CHECK, 'voices):')
print(', '.join(f'{w} {s:.2f}' for w, s in sorted(medians.items(), key=lambda kv: -kv[1])))
if medians[NAME] >= 0.6 and medians[NAME] - others[worst] >= 0.3:
    print(f'Looks usable: "{NAME}" scores {medians[NAME]:.2f}, and the closest other word ("{worst}") {others[worst]:.2f}.')
else:
    print(f'Not usable for this name: "{worst}" scores {others[worst]:.2f} against {medians[NAME]:.2f} for "{NAME}" itself, '
          'so the bot would react to it. Keep the file out of whisper/extras.')


In [ ]:
# 5. Download the model, and save it right away (Colab deletes its files when the
# session ends). If nothing downloads, use the Files panel on the left:
# output > <name> > <name>.onnx > right-click > Download.
from google.colab import files
files.download(f'output/{NAME}/{NAME}.onnx')
